In [1]:
import os.path as osp
from math import ceil

import torch
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import DenseGraphConv, GCNConv, dense_mincut_pool
from torch_geometric.utils import to_dense_adj, to_dense_batch

In [112]:
#path = osp.join(osp.dirname(osp.realpath(__file__)), '..', 'data', 'PROTEINS')

# Use the current working directory instead of `__file__` for jupyter or in interactive environment
path = osp.join(osp.abspath(''), '..', 'data', 'PROTEINS')

dataset = TUDataset(path, name='PROTEINS').shuffle()
average_nodes = int(dataset.data.x.size(0) / len(dataset))
n = (len(dataset) + 9) // 10
test_dataset = dataset[:n]
val_dataset = dataset[n:2 * n]
train_dataset = dataset[2 * n:]
test_loader = DataLoader(test_dataset, batch_size=20)
val_loader = DataLoader(val_dataset, batch_size=20)
train_loader = DataLoader(train_dataset, batch_size=20)

/home/iiitb/anaconda3/envs/SuperPixels/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The given 'InMemoryDataset' only references a subset of examples of the full dataset, but 'data' will contain information of the full dataset. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [70]:
dataset, len(dataset) # this means there are 1113 graphs are there

(PROTEINS(1113), 1113)

In [71]:
dataset.data

Data(x=[43471, 3], edge_index=[2, 162088], y=[1113])

In [72]:
dataset[0] # this would result information regarding the first set of graph

Data(edge_index=[2, 46], x=[14, 3], y=[1])

In [79]:
dataset.data # 43,471 nodes in total across all graphs (1113) in the dataset.

# edge_index Shape [2, 162088] means there are 162,088 edges in total.
# The first row contains the source nodes and second row contains the target nodes
# and each column represents an edge between two nodes

# y=[1113] represent graph output label for 1113 graphs we have

Data(x=[43471, 3], edge_index=[2, 162088], y=[1113])

In [74]:
dataset.data.x #here each row represent a node with feature of dim 3

tensor([[1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        ...,
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.]])

In [76]:
dataset.data.size(), dataset.data.x.size(), dataset.data.y.size(), dataset.data.x.size(0)

((43471, 43471), torch.Size([43471, 3]), torch.Size([1113]), 43471)

In [81]:
#train_loader.dataset.__getitem__(0)
train_dataset.__getitem__(3),train_dataset[0]

(Data(edge_index=[2, 230], x=[54, 3], y=[1]),
 Data(edge_index=[2, 186], x=[48, 3], y=[1]))

In [49]:
train_dataset[3].edge_index

tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5],
        [1, 2, 3, 0, 2, 3, 0, 1, 3, 4, 5, 0, 1, 2, 4, 5, 2, 3, 5, 2, 3, 4]])

In [50]:
len(train_loader.dataset), len(train_dataset), len(val_dataset), len(test_dataset)

(889, 889, 112, 112)

In [130]:
class Net(torch.nn.Module):
    def __init__(self, in_channels, out_channels, hidden_channels=32):
        super().__init__()

        # print(in_channels, out_channels, average_nodes)

        self.conv1 = GCNConv(in_channels, hidden_channels)
        num_nodes = ceil(0.5 * average_nodes)
        # print(num_nodes)
        self.pool1 = Linear(hidden_channels, num_nodes)

        self.conv2 = DenseGraphConv(hidden_channels, hidden_channels)
        num_nodes = ceil(0.5 * num_nodes)
        # print(num_nodes)
        self.pool2 = Linear(hidden_channels, num_nodes)

        self.conv3 = DenseGraphConv(hidden_channels, hidden_channels)

        self.lin1 = Linear(hidden_channels, hidden_channels)
        self.lin2 = Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()

        x, mask = to_dense_batch(x, batch)
        adj = to_dense_adj(edge_index, batch)

        s = self.pool1(x)
        x, adj, mc1, o1 = dense_mincut_pool(x, adj, s, mask)

        x = self.conv2(x, adj).relu()
        s = self.pool2(x)

        x, adj, mc2, o2 = dense_mincut_pool(x, adj, s)

        x = self.conv3(x, adj)

        x = x.mean(dim=1)
        x = self.lin1(x).relu()
        x = self.lin2(x)
        return F.log_softmax(x, dim=-1), mc1 + mc2, o1 + o2

In [131]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net(dataset.num_features, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

In [132]:
dataset

PROTEINS(1113)

In [133]:
dataset.num_features, dataset.num_classes

(3, 2)

In [134]:
train_loader.dataset.__getitem__(0)

Data(edge_index=[2, 98], x=[27, 3], y=[1])

In [135]:
# batch of 20
count = 0
for data in train_loader:
    # print(data)
    # break
    count +=1
print(count)

45


In [ ]:
#DataBatch(edge_index=[2, 2978], x=[783, 3], y=[20], batch=[783], ptr=[21])
# x=[783, 3] = features
# edge_index=[2, 2978] = num_of_class

In [136]:
def train(epoch):
    model.train()
    loss_all = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out, mc_loss, o_loss = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(out, data.y.view(-1)) + mc_loss + o_loss
        loss.backward()
        loss_all += data.y.size(0) * float(loss)
        optimizer.step()
    return loss_all / len(train_dataset)

In [137]:
@torch.no_grad()
def test(loader):
    model.eval()
    correct = 0
    loss_all = 0

    for data in loader:
        data = data.to(device)
        pred, mc_loss, o_loss = model(data.x, data.edge_index, data.batch)
        loss = F.nll_loss(pred, data.y.view(-1)) + mc_loss + o_loss
        loss_all += data.y.size(0) * float(loss)
        correct += int(pred.max(dim=1)[1].eq(data.y.view(-1)).sum())

    return loss_all / len(loader.dataset), correct / len(loader.dataset)

In [ ]:
times = []
best_val_acc = test_acc = 0
best_val_loss = float('inf')
patience = start_patience = 50
for epoch in range(1, 15000):
    train_loss = train(epoch)
    _, train_acc = test(train_loader)
    val_loss, val_acc = test(val_loader)
    if val_loss < best_val_loss:
        test_loss, test_acc = test(test_loader)
        best_val_acc = val_acc
        patience = start_patience
    else:
        patience -= 1
        if patience == 0:
            break
    print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, '
          f'Train Acc: {train_acc:.3f}, Val Loss: {val_loss:.3f}, '
          f'Val Acc: {val_acc:.3f}, Test Loss: {test_loss:.3f}, '
          f'Test Acc: {test_acc:.3f}')
    times.append(time.time() - start)
print(f"Median time per epoch: {torch.tensor(times).median():.4f}s")

Epoch: 001, Train Loss: 1.028, Train Acc: 0.667, Val Loss: 1.057, Val Acc: 0.643, Test Loss: 1.022, Test Acc: 0.705
Epoch: 002, Train Loss: 0.989, Train Acc: 0.727, Val Loss: 1.058, Val Acc: 0.661, Test Loss: 1.011, Test Acc: 0.732
Epoch: 003, Train Loss: 0.969, Train Acc: 0.742, Val Loss: 1.053, Val Acc: 0.661, Test Loss: 0.999, Test Acc: 0.741
Epoch: 004, Train Loss: 0.955, Train Acc: 0.744, Val Loss: 1.042, Val Acc: 0.679, Test Loss: 0.988, Test Acc: 0.750
Epoch: 005, Train Loss: 0.943, Train Acc: 0.755, Val Loss: 1.030, Val Acc: 0.696, Test Loss: 0.979, Test Acc: 0.759
Epoch: 006, Train Loss: 0.932, Train Acc: 0.756, Val Loss: 1.018, Val Acc: 0.688, Test Loss: 0.971, Test Acc: 0.768
Epoch: 007, Train Loss: 0.923, Train Acc: 0.762, Val Loss: 1.010, Val Acc: 0.688, Test Loss: 0.965, Test Acc: 0.768
Epoch: 008, Train Loss: 0.917, Train Acc: 0.764, Val Loss: 1.001, Val Acc: 0.688, Test Loss: 0.960, Test Acc: 0.759
Epoch: 009, Train Loss: 0.913, Train Acc: 0.765, Val Loss: 0.997, Val Ac